In [6]:
import polars as pl
from pathlib import Path
import json
import re
import pandas as pd

In [7]:
concept_path = Path("/home/q039tl/OpenICU.example/output/project/workspace/concept")
concept_dict_path = Path("/home/q039tl/ricu/inst/extdata/config/concept-dict.json")

In [8]:
with concept_dict_path.open("r", encoding="utf-8") as f:
    concept_dict = json.load(f)

In [12]:
def snake_case(value: str | None) -> str | None:
    if value is None:
        return None

    value = value.strip()
    value = value.replace("%", "percent")
    value = value.replace("+", "plus")
    value = value.replace("/", " per ")
    value = value.replace("-", " ")
    value = re.sub(r"[^\w\s]", "", value)
    value = re.sub(r"\s+", "_", value)
    value = value.lower().strip("_")
    return value


rows = []

for abbr, concept in concept_dict.items():
    description = concept.get("description")
    category = concept.get("category")
    unit = concept.get("unit")
    sources = concept.get("sources", {})

    rows.append(
        {
            "abbr": abbr,
            "description": description,
            "definition_name": snake_case(description),
            "category": category,
            "unit": json.dumps(unit) if isinstance(unit, (list, dict)) else unit,
            "sources": list(sources.keys()),
            "has_miiv": "miiv" in sources,
            "has_mimic": "mimic" in sources,
            "has_eicu": "eicu" in sources,
        }
    )

ricu_concepts = pl.DataFrame(rows)

ricu_concepts.head()

abbr,description,definition_name,category,unit,sources,has_miiv,has_mimic,has_eicu
str,str,str,str,str,list[str],bool,bool,bool
"""abx""","""antibiotics""","""antibiotics""","""medications""",null,"[""aumc"", ""eicu"", … ""sic""]",true,true,true
"""adh_rate""","""vasopressin rate""","""vasopressin_rate""","""medications""","""[""units/min"", ""U/min""]""","[""aumc"", ""eicu"", … ""mimic_demo""]",true,true,true
"""adm""","""patient admission type""","""patient_admission_type""","""demographics""",null,"[""aumc"", ""eicu"", … ""mimic_demo""]",true,true,true
"""age""","""patient age""","""patient_age""","""demographics""","""years""","[""aumc"", ""eicu"", … ""sic""]",true,true,true
"""alb""","""albumin""","""albumin""","""chemistry""","""g/dL""","[""aumc"", ""eicu"", … ""sic""]",true,true,true


In [18]:
yaib_abbrs = [
    "alb", "alp", "alt", "ast", "be", "bicar", "bili", "bili_dir",
                  "bnd", "bun", "ca", "cai", "ck", "ckmb", "cl", "crea", "crp", 
                  "dbp", "fgn", "fio2", "glu", "hgb", "hr", "inr_pt", "k", "lact",
                  "lymph", "map", "mch", "mchc", "mcv", "methb", "mg", "na", "neut", 
                  "o2sat", "pco2", "ph", "phos", "plt", "po2", "ptt", "resp", "sbp", 
                  "temp", "tnt", "urine", "wbc"
]

yaib_df = pl.DataFrame({"abbr": yaib_abbrs})

In [19]:
yaib_df

abbr
str
"""alb"""
"""alp"""
"""alt"""
"""ast"""
"""be"""
…
"""sbp"""
"""temp"""
"""tnt"""


In [20]:
mapped = (
    yaib_df
    .join(ricu_concepts, on="abbr", how="left")
    .with_columns(
        pl.col("description").is_null().alias("missing_in_ricu")
    )
)

mapped

abbr,description,definition_name,category,unit,sources,has_miiv,has_mimic,has_eicu,missing_in_ricu
str,str,str,str,str,list[str],bool,bool,bool,bool
"""alb""","""albumin""","""albumin""","""chemistry""","""g/dL""","[""aumc"", ""eicu"", … ""sic""]",true,true,true,false
"""alp""","""alkaline phosphatase""","""alkaline_phosphatase""","""chemistry""","""[""IU/L"", ""U/l""]""","[""aumc"", ""eicu"", … ""sic""]",true,true,true,false
"""alt""","""alanine aminotransferase""","""alanine_aminotransferase""","""chemistry""","""[""IU/L"", ""U/l""]""","[""aumc"", ""eicu"", … ""sic""]",true,true,true,false
"""ast""","""aspartate aminotransferase""","""aspartate_aminotransferase""","""chemistry""","""[""IU/L"", ""U/l""]""","[""aumc"", ""eicu"", … ""sic""]",true,true,true,false
"""be""","""base excess""","""base_excess""","""blood gas""","""[""mEq/L"", ""mmol/l""]""","[""aumc"", ""eicu"", … ""sic""]",true,true,true,false
…,…,…,…,…,…,…,…,…,…
"""sbp""","""systolic blood pressure""","""systolic_blood_pressure""","""vitals""","""[""mmHg"", ""mm Hg""]""","[""aumc"", ""eicu"", … ""sic""]",true,true,true,false
"""temp""","""temperature""","""temperature""","""vitals""","""[""C"", ""\u00b0C""]""","[""aumc"", ""eicu"", … ""sic""]",true,true,true,false
"""tnt""","""troponin t""","""troponin_t""","""chemistry""","""ng/mL""","[""aumc"", ""eicu"", … ""sic""]",true,true,true,false


In [21]:
mapped.select(
    [
        "abbr",
        "definition_name",
        "description",
        "category",
        "unit",
        "has_miiv",
        "has_eicu",
        "missing_in_ricu",
    ]
)

abbr,definition_name,description,category,unit,has_miiv,has_eicu,missing_in_ricu
str,str,str,str,str,bool,bool,bool
"""alb""","""albumin""","""albumin""","""chemistry""","""g/dL""",true,true,false
"""alp""","""alkaline_phosphatase""","""alkaline phosphatase""","""chemistry""","""[""IU/L"", ""U/l""]""",true,true,false
"""alt""","""alanine_aminotransferase""","""alanine aminotransferase""","""chemistry""","""[""IU/L"", ""U/l""]""",true,true,false
"""ast""","""aspartate_aminotransferase""","""aspartate aminotransferase""","""chemistry""","""[""IU/L"", ""U/l""]""",true,true,false
"""be""","""base_excess""","""base excess""","""blood gas""","""[""mEq/L"", ""mmol/l""]""",true,true,false
…,…,…,…,…,…,…,…
"""sbp""","""systolic_blood_pressure""","""systolic blood pressure""","""vitals""","""[""mmHg"", ""mm Hg""]""",true,true,false
"""temp""","""temperature""","""temperature""","""vitals""","""[""C"", ""\u00b0C""]""",true,true,false
"""tnt""","""troponin_t""","""troponin t""","""chemistry""","""ng/mL""",true,true,false


In [22]:
missing = mapped.filter(pl.col("missing_in_ricu"))

missing.select("abbr")

abbr
str


In [24]:
mapped.select(
    [
        "abbr",
        "definition_name",
        "description",
        "category",
        "unit",
        "has_miiv",
        "has_eicu",
    ]
).write_csv("yaib_ricu_concept_mapping.csv")

In [25]:
mapped.write_parquet("yaib_ricu_concept_mapping.parquet")

In [26]:
excluded_abbrs = ["spo2"]

mapped_filtered = mapped.filter(
    ~pl.col("abbr").is_in(excluded_abbrs)
)

mapped_filtered

abbr,description,definition_name,category,unit,sources,has_miiv,has_mimic,has_eicu,missing_in_ricu
str,str,str,str,str,list[str],bool,bool,bool,bool
"""alb""","""albumin""","""albumin""","""chemistry""","""g/dL""","[""aumc"", ""eicu"", … ""sic""]",true,true,true,false
"""alp""","""alkaline phosphatase""","""alkaline_phosphatase""","""chemistry""","""[""IU/L"", ""U/l""]""","[""aumc"", ""eicu"", … ""sic""]",true,true,true,false
"""alt""","""alanine aminotransferase""","""alanine_aminotransferase""","""chemistry""","""[""IU/L"", ""U/l""]""","[""aumc"", ""eicu"", … ""sic""]",true,true,true,false
"""ast""","""aspartate aminotransferase""","""aspartate_aminotransferase""","""chemistry""","""[""IU/L"", ""U/l""]""","[""aumc"", ""eicu"", … ""sic""]",true,true,true,false
"""be""","""base excess""","""base_excess""","""blood gas""","""[""mEq/L"", ""mmol/l""]""","[""aumc"", ""eicu"", … ""sic""]",true,true,true,false
…,…,…,…,…,…,…,…,…,…
"""sbp""","""systolic blood pressure""","""systolic_blood_pressure""","""vitals""","""[""mmHg"", ""mm Hg""]""","[""aumc"", ""eicu"", … ""sic""]",true,true,true,false
"""temp""","""temperature""","""temperature""","""vitals""","""[""C"", ""\u00b0C""]""","[""aumc"", ""eicu"", … ""sic""]",true,true,true,false
"""tnt""","""troponin t""","""troponin_t""","""chemistry""","""ng/mL""","[""aumc"", ""eicu"", … ""sic""]",true,true,true,false


In [27]:
name_overrides = {
    "abx": "antibiotics",
    "o2sat": "oxygen_saturation",
    "sao2": "arterial_oxygen_saturation",
    "urine": "urine_output",
    "dbp": "diastolic_blood_pressure",
    "sbp": "systolic_blood_pressure",
    "hr": "heart_rate",
    "crea": "creatinine",
}

overrides_df = pl.DataFrame(
    {
        "abbr": list(name_overrides.keys()),
        "openicu_name_override": list(name_overrides.values()),
    }
)

mapped = (
    mapped
    .join(overrides_df, on="abbr", how="left")
    .with_columns(
        pl.coalesce(
            ["openicu_name_override", "definition_name"]
        ).alias("openicu_concept_name")
    )
)

In [28]:
mapped.select(
    [
        "abbr",
        "openicu_concept_name",
        "definition_name",
        "description",
        "category",
        "unit",
        "has_miiv",
        "has_eicu",
        "missing_in_ricu",
    ]
)

abbr,openicu_concept_name,definition_name,description,category,unit,has_miiv,has_eicu,missing_in_ricu
str,str,str,str,str,str,bool,bool,bool
"""alb""","""albumin""","""albumin""","""albumin""","""chemistry""","""g/dL""",true,true,false
"""alp""","""alkaline_phosphatase""","""alkaline_phosphatase""","""alkaline phosphatase""","""chemistry""","""[""IU/L"", ""U/l""]""",true,true,false
"""alt""","""alanine_aminotransferase""","""alanine_aminotransferase""","""alanine aminotransferase""","""chemistry""","""[""IU/L"", ""U/l""]""",true,true,false
"""ast""","""aspartate_aminotransferase""","""aspartate_aminotransferase""","""aspartate aminotransferase""","""chemistry""","""[""IU/L"", ""U/l""]""",true,true,false
"""be""","""base_excess""","""base_excess""","""base excess""","""blood gas""","""[""mEq/L"", ""mmol/l""]""",true,true,false
…,…,…,…,…,…,…,…,…
"""sbp""","""systolic_blood_pressure""","""systolic_blood_pressure""","""systolic blood pressure""","""vitals""","""[""mmHg"", ""mm Hg""]""",true,true,false
"""temp""","""temperature""","""temperature""","""temperature""","""vitals""","""[""C"", ""\u00b0C""]""",true,true,false
"""tnt""","""troponin_t""","""troponin_t""","""troponin t""","""chemistry""","""ng/mL""",true,true,false


In [29]:
from pathlib import Path
import polars as pl

concept_path = Path("/home/q039tl/OpenICU.example/output/project/workspace/concept")

mimic_files = sorted(concept_path.glob("*/1.0.0/mimic-iv.parquet"))

print("number of mimic concept files:", len(mimic_files))
print(mimic_files[:5])

number of mimic concept files: 86
[PosixPath('/home/q039tl/OpenICU.example/output/project/workspace/concept/CO2_partial_pressure/1.0.0/mimic-iv.parquet'), PosixPath('/home/q039tl/OpenICU.example/output/project/workspace/concept/C_reactive_protein/1.0.0/mimic-iv.parquet'), PosixPath('/home/q039tl/OpenICU.example/output/project/workspace/concept/GCS_eye/1.0.0/mimic-iv.parquet'), PosixPath('/home/q039tl/OpenICU.example/output/project/workspace/concept/GCS_motor/1.0.0/mimic-iv.parquet'), PosixPath('/home/q039tl/OpenICU.example/output/project/workspace/concept/GCS_verbal/1.0.0/mimic-iv.parquet')]


In [30]:
rows = []

for file in mimic_files:
    concept_name = file.parts[-3]   # concept/<concept_name>/1.0.0/mimic-iv.parquet

    df = pl.read_parquet(file)

    rows.append(
        {
            "concept_name": concept_name,
            "path": str(file),
            "n_rows": df.height,
            "columns": df.columns,
            "code_n_unique": df.select(pl.col("code").n_unique()).item() if "code" in df.columns else None,
            "table_n_unique": df.select(pl.col("table").n_unique()).item() if "table" in df.columns else None,
        }
    )

concept_overview = pl.DataFrame(rows)

concept_overview.sort("concept_name")

concept_name,path,n_rows,columns,code_n_unique,table_n_unique
str,str,i64,list[str],i64,i64
"""CO2_partial_pressure""","""/home/q039tl/OpenICU.example/o…",698217,"[""subject_id"", ""time"", … ""table""]",1,1
"""C_reactive_protein""","""/home/q039tl/OpenICU.example/o…",178039,"[""subject_id"", ""time"", … ""table""]",1,1
"""GCS_eye""","""/home/q039tl/OpenICU.example/o…",2209510,"[""subject_id"", ""time"", … ""table""]",1,1
"""GCS_motor""","""/home/q039tl/OpenICU.example/o…",2199619,"[""subject_id"", ""time"", … ""table""]",1,1
"""GCS_verbal""","""/home/q039tl/OpenICU.example/o…",2205121,"[""subject_id"", ""time"", … ""table""]",1,1
…,…,…,…,…,…
"""troponin_I""","""/home/q039tl/OpenICU.example/o…",0,"[""subject_id"", ""time"", … ""table""]",0,0
"""troponin_t""","""/home/q039tl/OpenICU.example/o…",459872,"[""subject_id"", ""time"", … ""table""]",1,1
"""urine_output""","""/home/q039tl/OpenICU.example/o…",4251397,"[""subject_id"", ""time"", … ""table""]",1,1
